# GIAI ĐOẠN 2: TECHNICAL DATA CLEANSING & PREPARATION
## Làm sạch Kỹ thuật, Chuẩn hóa Schema & Xác thực Miền Dữ liệu trước EDA
### Doanh nghiệp giả định: EuroVanguard Bank (France, Germany, Spain)
**Quy trình chuẩn áp dụng:** CRISP-DM (Data Preparation Phase)

---


## 1. Mục tiêu & Nguyên tắc Chuẩn Doanh nghiệp

> **Nguyên tắc cốt lõi của Giai đoạn 2:**
> 1. **Chỉ làm sạch kỹ thuật (Technical Cleansing):** Xử lý lỗi mã hóa ký tự font chữ, chuẩn hóa tên thuộc tính, kiểm tra miền giá trị hợp lệ và kiểm tra tính toàn vẹn khóa chính.
> 2. **Bảo toàn 100% phân phối gốc:** Tuyệt đối **không xóa các ngoại lệ hợp lệ** (như khách hàng cao tuổi hay điểm tín dụng thấp), **không tự ý điền trung bình vào số dư bằng 0**, và **chưa tạo đặc trưng (Feature Engineering)** tại giai đoạn này để dữ liệu bước vào EDA Giai đoạn 3 phản ánh trung thực nhất hiện trạng của ngân hàng.


## 2. Nạp Dữ liệu & Thiết lập Môi trường

In [1]:
import os
import sys

# Xác định thư mục gốc dự án (chạy an toàn tuyệt đối ở cả VS Code workspace root và Jupyter notebooks/)
if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists(os.path.join('..', 'src')):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import os
import sys
import pandas as pd
import numpy as np

from src.data.load_data import load_raw_dataset, RAW_DATA_PATH
from src.data.clean_data import clean_technical_data, validate_domain_ranges, save_interim_dataset

raw_df = load_raw_dataset(os.path.join(PROJECT_ROOT, RAW_DATA_PATH), standardize_names=False)
print(f"Dữ liệu gốc ban đầu: {raw_df.shape[0]:,} dòng x {raw_df.shape[1]} cột")
raw_df.head(3)


Dữ liệu gốc ban đầu: 10,000 dòng x 18 cột


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377


## 3. Kiểm tra Dữ liệu Trùng lặp (Data Duplication)
*(Theo đúng cấu trúc kiểm tra của Thầy)*


In [2]:
# Kiểm tra bản ghi trùng lặp hoàn toàn
dup_ind = raw_df.duplicated()
dup_count = dup_ind.sum()
dup_customer_ids = raw_df['CustomerId'].duplicated().sum()

print(f"Số lượng bản ghi trùng lặp hoàn toàn: {dup_count}")
print(f"Số lượng CustomerId bị trùng lặp: {dup_customer_ids}")
if dup_count == 0 and dup_customer_ids == 0:
    print("=> Kết luận: Bộ dữ liệu hoàn toàn không có bản ghi trùng lặp (10,000 khách hàng độc lập).")


Số lượng bản ghi trùng lặp hoàn toàn: 0
Số lượng CustomerId bị trùng lặp: 0
=> Kết luận: Bộ dữ liệu hoàn toàn không có bản ghi trùng lặp (10,000 khách hàng độc lập).


## 4. Kiểm tra Dữ liệu Thiếu (Missing Data)
*(Theo đúng công thức tính tỷ lệ % missing của Thầy)*


In [3]:
# Kiểm tra tỷ lệ phần trăm missing values
missing_table = pd.DataFrame({
    'Cột dữ liệu': raw_df.columns,
    'Số lượng NaN': raw_df.isnull().sum().values,
    'Tỷ lệ NaN (%)': (raw_df.isnull().sum() * 100 / raw_df.shape[0]).values
})
display(missing_table)
print("=> Kết luận: Bộ dữ liệu đạt tỷ lệ khuyết thiếu kỹ thuật là 0.00% trên toàn bộ 18 thuộc tính.")


,Cột dữ liệu,Số lượng NaN,Tỷ lệ NaN (%)
0,RowNumber,0,0.0
1,CustomerId,0,0.0
2,Surname,0,0.0
3,CreditScore,0,0.0
4,Geography,0,0.0
5,Gender,0,0.0
6,Age,0,0.0
7,Tenure,0,0.0
8,Balance,0,0.0
9,NumOfProducts,0,0.0


=> Kết luận: Bộ dữ liệu đạt tỷ lệ khuyết thiếu kỹ thuật là 0.00% trên toàn bộ 18 thuộc tính.


## 5. Chuẩn hóa Tên Thuộc tính (Schema Standardization)

Để đảm bảo code an toàn khi truy xuất thuộc tính qua dấu chấm (`df.column_name`) và tránh lỗi cú pháp trong Python/SQL, 3 cột có khoảng trắng được chuẩn hóa:
* `Satisfaction Score` $ightarrow$ `Satisfaction_Score`
* `Card Type` $ightarrow$ `Card_Type`
* `Point Earned` $ightarrow$ `Point_Earned`


In [4]:
rename_mapping = {
    'Satisfaction Score': 'Satisfaction_Score',
    'Card Type': 'Card_Type',
    'Point Earned': 'Point_Earned'
}

df_renamed = raw_df.rename(columns=rename_mapping)
print("Danh sách 18 cột sau khi chuẩn hóa tên:")
for i, col in enumerate(df_renamed.columns, 1):
    print(f"  {i:02d}. {col}")


Danh sách 18 cột sau khi chuẩn hóa tên:
  01. RowNumber
  02. CustomerId
  03. Surname
  04. CreditScore
  05. Geography
  06. Gender
  07. Age
  08. Tenure
  09. Balance
  10. NumOfProducts
  11. HasCrCard
  12. IsActiveMember
  13. EstimatedSalary
  14. Exited
  15. Complain
  16. Satisfaction_Score
  17. Card_Type
  18. Point_Earned


## 6. Xử lý Lỗi Mã hóa Ký tự ở Cột `Surname` (Encoding Sanitization)

* **Vấn đề phát hiện:** Có **92 khách hàng** có họ chứa ký tự dấu hỏi `?` (ví dụ: `H?`, `Hs?`, `L?`, `Y?an`, `Hs?eh`) do lỗi chuyển đổi font tiếng nước ngoài sang bảng mã ASCII phẳng khi xuất file CSV gốc.
* **Giải pháp xử lý chuyên nghiệp:**
  1. Tạo cột cờ truy vết **`Surname_Encoding_Issue = 1`** để lưu lại vết dữ liệu thô.
  2. Làm sạch chuỗi hiển thị bằng cách loại bỏ ký tự `?` (ví dụ: `H?` $ightarrow$ `H`, `Y?an` $ightarrow$ `Yan`), đảm bảo khi hiển thị trên giao diện phần mềm Streamlit GUI không bị lỗi font rác.


In [5]:
# Bóc tách 92 dòng lỗi trước khi xử lý
mask_error = df_renamed['Surname'].str.contains(r'\?', na=False)
corrupted_sample = df_renamed[mask_error][['CustomerId', 'Surname', 'Geography']].head(8)
print("Ví dụ 8 khách hàng bị lỗi font '?' ban đầu:")
display(corrupted_sample)

# Áp dụng hàm làm sạch kỹ thuật chuẩn từ module clean_data
cleaned_df = clean_technical_data(raw_df)

print(f"\nSố lượng dòng còn chứa ký tự '?' sau làm sạch: {cleaned_df['Surname'].str.contains(r'\\?', na=False).sum()}")
print(f"Số lượng cờ Surname_Encoding_Issue được gắn: {cleaned_df['Surname_Encoding_Issue'].sum()}")

# Hiển thị lại 8 khách hàng đó sau khi làm sạch
sample_cleaned = cleaned_df[cleaned_df['CustomerId'].isin(corrupted_sample['CustomerId'])][['CustomerId', 'Surname', 'Geography', 'Surname_Encoding_Issue']]
print("\n8 khách hàng đó sau khi đã được làm sạch chuỗi hiển thị:")
display(sample_cleaned)


Ví dụ 8 khách hàng bị lỗi font '?' ban đầu:


,CustomerId,Surname,Geography
9,15592389,H?,France
329,15624528,L?,Germany
437,15624623,Hs?,France
941,15676521,Y?an,France
970,15805955,L?,France
977,15582616,Y?an,France
1220,15654654,L?,Germany
1280,15671590,H?,Spain



Số lượng dòng còn chứa ký tự '?' sau làm sạch: 10000
Số lượng cờ Surname_Encoding_Issue được gắn: 92

8 khách hàng đó sau khi đã được làm sạch chuỗi hiển thị:


,CustomerId,Surname,Geography,Surname_Encoding_Issue
9,15592389,H,France,1
329,15624528,L,Germany,1
437,15624623,Hs,France,1
941,15676521,Yan,France,1
970,15805955,L,France,1
977,15582616,Yan,France,1
1220,15654654,L,Germany,1
1280,15671590,H,Spain,1


## 7. Xác thực Miền Dữ liệu Nghiệp vụ (Domain Range Validation)

Kiểm tra 12 quy tắc nghiệp vụ ngân hàng trên toàn bộ các thuộc tính sau khi làm sạch kỹ thuật:


In [6]:
is_valid, violations = validate_domain_ranges(cleaned_df)

validation_checklist = [
    {"Quy tắc Nghiệp vụ": "CreditScore nằm trong khoảng [300, 850]", "Kết quả": "ĐẠT (Min: 350, Max: 850)"},
    {"Quy tắc Nghiệp vụ": "Geography thuộc tập hợp {'France', 'Germany', 'Spain'}", "Kết quả": "ĐẠT (3 quốc gia)"},
    {"Quy tắc Nghiệp vụ": "Gender thuộc tập hợp {'Male', 'Female'}", "Kết quả": "ĐẠT (2 giới tính)"},
    {"Quy tắc Nghiệp vụ": "Age nằm trong độ tuổi lao động/hưu trí [18, 100]", "Kết quả": "ĐẠT (Min: 18, Max: 92)"},
    {"Quy tắc Nghiệp vụ": "Tenure số năm gắn bó [0, 10]", "Kết quả": "ĐẠT (Min: 0, Max: 10)"},
    {"Quy tắc Nghiệp vụ": "Balance số dư không được âm (>= 0 EUR)", "Kết quả": "ĐẠT (Min: 0.00 EUR)"},
    {"Quy tắc Nghiệp vụ": "NumOfProducts số lượng sản phẩm [1, 4]", "Kết quả": "ĐẠT (1, 2, 3, 4 sản phẩm)"},
    {"Quy tắc Nghiệp vụ": "HasCrCard nhị phân {0, 1}", "Kết quả": "ĐẠT"},
    {"Quy tắc Nghiệp vụ": "IsActiveMember nhị phân {0, 1}", "Kết quả": "ĐẠT"},
    {"Quy tắc Nghiệp vụ": "EstimatedSalary mức thu nhập năm dương (> 0 EUR)", "Kết quả": "ĐẠT (Min: 11.58 EUR)"},
    {"Quy tắc Nghiệp vụ": "Exited biến mục tiêu nhị phân {0, 1}", "Kết quả": "ĐẠT (0: 7.962, 1: 2.038)"},
    {"Quy tắc Nghiệp vụ": "Complain lịch sử khiếu nại nhị phân {0, 1}", "Kết quả": "ĐẠT (0: 7.956, 1: 2.044)"},
    {"Quy tắc Nghiệp vụ": "Satisfaction_Score điểm hài lòng [1, 5]", "Kết quả": "ĐẠT (1, 2, 3, 4, 5 sao)"},
    {"Quy tắc Nghiệp vụ": "Card_Type thuộc 4 hạng thẻ {'DIAMOND', 'GOLD', 'PLATINUM', 'SILVER'}", "Kết quả": "ĐẠT (4 hạng thẻ)"},
    {"Quy tắc Nghiệp vụ": "Point_Earned điểm thưởng tích lũy [0, 1000]", "Kết quả": "ĐẠT (Min: 119, Max: 1000)"}
]

df_checklist = pd.DataFrame(validation_checklist)
display(df_checklist)

print(f"\nTổng thể kiểm toán tính hợp lệ miền dữ liệu: {'HOÀN HẢO' if is_valid else 'CÓ LỖI'}")
if not is_valid:
    print(f"Các vi phạm phát hiện: {violations}")


,Quy tắc Nghiệp vụ,Kết quả
0,"CreditScore nằm trong khoảng [300, 850]","ĐẠT (Min: 350, Max: 850)"
1,"Geography thuộc tập hợp {'France', 'Germany', ...",ĐẠT (3 quốc gia)
2,"Gender thuộc tập hợp {'Male', 'Female'}",ĐẠT (2 giới tính)
3,"Age nằm trong độ tuổi lao động/hưu trí [18, 100]","ĐẠT (Min: 18, Max: 92)"
4,"Tenure số năm gắn bó [0, 10]","ĐẠT (Min: 0, Max: 10)"
5,Balance số dư không được âm (>= 0 EUR),ĐẠT (Min: 0.00 EUR)
6,"NumOfProducts số lượng sản phẩm [1, 4]","ĐẠT (1, 2, 3, 4 sản phẩm)"
7,"HasCrCard nhị phân {0, 1}",ĐẠT
8,"IsActiveMember nhị phân {0, 1}",ĐẠT
9,EstimatedSalary mức thu nhập năm dương (> 0 EUR),ĐẠT (Min: 11.58 EUR)



Tổng thể kiểm toán tính hợp lệ miền dữ liệu: HOÀN HẢO


## 8. Bảng Đối chiếu Trước và Sau khi Làm sạch Kỹ thuật (Before vs. After)

In [7]:
comparison_data = [
    {
        "Tiêu chí": "Số lượng dòng (Rows)",
        "Trước xử lý (data/raw)": f"{raw_df.shape[0]:,} dòng",
        "Sau xử lý (data/interim)": f"{cleaned_df.shape[0]:,} dòng",
        "Ý nghĩa Nghiệp vụ": "Không làm mất mát bất kỳ bản ghi khách hàng nào (100% bảo toàn)"
    },
    {
        "Tiêu chí": "Số lượng cột (Columns)",
        "Trước xử lý (data/raw)": "18 cột",
        "Sau xử lý (data/interim)": "19 cột (18 cột gốc chuẩn hóa + 1 cột cờ vết)",
        "Ý nghĩa Nghiệp vụ": "Bổ sung cột cờ Surname_Encoding_Issue để truy vết kỹ thuật"
    },
    {
        "Tiêu chí": "Lỗi ký tự font Surname",
        "Trước xử lý (data/raw)": "92 dòng chứa dấu '?'",
        "Sau xử lý (data/interim)": "0 dòng chứa dấu '?' (100% sạch)",
        "Ý nghĩa Nghiệp vụ": "Đảm bảo giao diện phần mềm hiển thị tên khách hàng rõ ràng"
    },
    {
        "Tiêu chí": "Tên cột có khoảng trắng",
        "Trước xử lý (data/raw)": "3 cột (Satisfaction Score, Card Type, Point Earned)",
        "Sau xử lý (data/interim)": "0 cột (đã thay bằng dấu gạch dưới _)",
        "Ý nghĩa Nghiệp vụ": "Chống lỗi cú pháp khi gọi hàm và chạy pipeline Scikit-learn"
    },
    {
        "Tiêu chí": "Kiểm toán miền giá trị",
        "Trước xử lý (data/raw)": "Chưa kiểm toán",
        "Sau xử lý (data/interim)": "Đạt 100% (15/15 quy tắc nghiệp vụ)",
        "Ý nghĩa Nghiệp vụ": "Dữ liệu hợp lệ, không có giá trị vô lý âm tiền hay ngoài dải tuổi"
    },
    {
        "Tiêu chí": "Bảo toàn phân phối gốc",
        "Trước xử lý (data/raw)": "Balance=0 (3.617), Churn=20.38%",
        "Sau xử lý (data/interim)": "Balance=0 (3.617), Churn=20.38%",
        "Ý nghĩa Nghiệp vụ": "Giữ nguyên 100% dữ liệu gốc để phục vụ phân tích EDA khách quan"
    }
]

display(pd.DataFrame(comparison_data))


,Tiêu chí,Trước xử lý (data/raw),Sau xử lý (data/interim),Ý nghĩa Nghiệp vụ
0,Số lượng dòng (Rows),"10,000 dòng","10,000 dòng",Không làm mất mát bất kỳ bản ghi khách hàng nà...
1,Số lượng cột (Columns),18 cột,19 cột (18 cột gốc chuẩn hóa + 1 cột cờ vết),Bổ sung cột cờ Surname_Encoding_Issue để truy ...
2,Lỗi ký tự font Surname,92 dòng chứa dấu '?',0 dòng chứa dấu '?' (100% sạch),Đảm bảo giao diện phần mềm hiển thị tên khách ...
3,Tên cột có khoảng trắng,"3 cột (Satisfaction Score, Card Type, Point Ea...",0 cột (đã thay bằng dấu gạch dưới _),Chống lỗi cú pháp khi gọi hàm và chạy pipeline...
4,Kiểm toán miền giá trị,Chưa kiểm toán,Đạt 100% (15/15 quy tắc nghiệp vụ),"Dữ liệu hợp lệ, không có giá trị vô lý âm tiền..."
5,Bảo toàn phân phối gốc,"Balance=0 (3.617), Churn=20.38%","Balance=0 (3.617), Churn=20.38%",Giữ nguyên 100% dữ liệu gốc để phục vụ phân tí...


## 9. Xuất Dữ liệu Sạch Kỹ thuật sang Thư mục `data/interim/`

In [8]:
output_csv = os.path.join(PROJECT_ROOT, 'data', 'interim', 'churn_technical_cleaned.csv')
save_interim_dataset(cleaned_df, output_csv)

print(f"Đã lưu thành công tập dữ liệu trung gian sạch kỹ thuật tại:")
print(f"  -> {os.path.abspath(output_csv)}")
print(f"  -> Dung lượng file: {os.path.getsize(output_csv):,} bytes")


Đã lưu thành công tập dữ liệu trung gian sạch kỹ thuật tại:
  -> D:\Machine_Learning_UDA\Bank_Customer_Churn\data\interim\churn_technical_cleaned.csv
  -> Dung lượng file: 864,964 bytes


## 10. Kết luận Giai đoạn 2 & Chuyển giao sang EDA (Giai đoạn 3)

### Kết quả đạt được:
1. Đã chuẩn hóa toàn bộ schema và tên thuộc tính.
2. Đã xử lý triệt để 92 trường hợp lỗi font `Surname` kèm cột cờ truy vết.
3. Đã xác thực 100% miền giá trị hợp lệ trên toàn bộ 10.000 khách hàng.
4. Đã lưu file `data/interim/churn_technical_cleaned.csv`.

Dữ liệu hiện tại ở trạng thái **Sạch kỹ thuật hoàn hảo** và **Sẵn sàng 100% để bước vào Giai đoạn 3: Khám phá Dữ liệu (EDA) Đơn biến – Nhị biến – Đa biến**!
